PyTorch — Commands and Usage in Machine Learning

1.  Tensor Creation
2.  Tensor Attributes & Type Conversion
3.  Tensor Operations
4.  Indexing, Slicing & Joining
5.  Autograd — Automatic Differentiation
6.  Building Models with nn.Module
7.  Loss Functions
8.  Optimizers
9.  Training Loop
10. GPU Usage
11. Common Layers & Activations

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

1. Tensor Creation
Tensors are PyTorch's core data structure — like NumPy arrays but GPU-capable and autodiff-aware.

In [ ]:
# From data
a = torch.tensor([1, 2, 3])                          # infer dtype from data
b = torch.tensor([[1.0, 2.0], [3.0, 4.0]])           # 2D float
c = torch.tensor([1, 2, 3], dtype=torch.float32)     # explicit dtype

# Filled tensors
zeros  = torch.zeros(3, 4)
ones   = torch.ones(2, 3)
full   = torch.full((2, 2), 7.0)
eye    = torch.eye(3)
empty  = torch.empty(2, 2)                           # uninitialized

# Ranges
arange  = torch.arange(0, 10, 2)                     # [0, 2, 4, 6, 8]
linspace = torch.linspace(0, 1, 5)                   # 5 values from 0 to 1

# Random
rand    = torch.rand(2, 3)                           # uniform [0, 1)
randn   = torch.randn(2, 3)                          # N(0, 1)
randint = torch.randint(0, 10, (2, 3))               # random ints

# From NumPy (shares memory when on CPU)
arr = np.array([1.0, 2.0, 3.0])
from_np = torch.from_numpy(arr)

print(f"tensor from list:  {a}")
print(f"zeros (3x4):\n{zeros}")
print(f"rand (2x3):\n{rand}")
print(f"from numpy:        {from_np}")

2. Tensor Attributes & Type Conversion
Inspect and convert tensors — shape, dtype, and device are the three you check most.

In [ ]:
x = torch.randn(3, 4)

print(f"shape:   {x.shape}")
print(f"size():  {x.size()}")
print(f"ndim:    {x.ndim}")
print(f"dtype:   {x.dtype}")
print(f"device:  {x.device}")
print(f"numel:   {x.numel()}")       # total number of elements

# Type conversion
x_f16 = x.to(torch.float16)          # half precision (GPU training)
x_i32 = x.to(torch.int32)
x_f64 = x.double()                   # shorthand for float64
x_f32 = x.float()                    # shorthand for float32 (most common in ML)

print(f"\nfloat16: {x_f16.dtype}")
print(f"float32: {x_f32.dtype}")
print(f"int32:   {x_i32.dtype}")

# To/from numpy
as_np   = x.numpy()                  # tensor → numpy (CPU only)
back    = torch.from_numpy(as_np)    # numpy → tensor
print(f"\nnumpy dtype: {as_np.dtype}")

# Scalar extraction
s = torch.tensor(3.14)
print(f"scalar .item(): {s.item()}")

3. Tensor Operations
Math, comparison, and reduction — all GPU-accelerated.

In [ ]:
a = torch.tensor([1.0, 4.0, 9.0])
b = torch.tensor([2.0, 2.0, 3.0])
A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
B = torch.tensor([[5.0, 6.0], [7.0, 8.0]])

# Element-wise
print(f"a + b:       {a + b}")
print(f"a * b:       {a * b}")
print(f"a ** 2:      {a ** 2}")
print(f"torch.sqrt:  {torch.sqrt(a)}")
print(f"torch.exp:   {torch.exp(b)}")
print(f"torch.log:   {torch.log(a)}")
print(f"torch.abs:   {torch.abs(torch.tensor([-1.0, 2.0, -3.0]))}")

# Matrix ops
print(f"\nA @ B:\n{torch.matmul(A, B)}")
print(f"A * B (elem):\n{A * B}")
print(f"A.T:\n{A.T}")
print(f"torch.dot(a,b): {torch.dot(a, b)}")

# Reductions
x = torch.arange(1.0, 7.0).reshape(2, 3)
print(f"\nx:\n{x}")
print(f"sum:      {x.sum()}")
print(f"sum dim=0:{x.sum(dim=0)}")
print(f"sum dim=1:{x.sum(dim=1)}")
print(f"mean:     {x.mean():.4f}")
print(f"max:      {x.max()}")
print(f"argmax:   {x.argmax()}")
print(f"argmax d=1:{x.argmax(dim=1)}")

4. Indexing, Slicing & Joining
Navigate and combine tensors — same as NumPy but with extra GPU awareness.

In [ ]:
x = torch.arange(1.0, 10.0).reshape(3, 3)
print(f"x:\n{x}")

# Indexing & slicing
print(f"x[0, 1]:    {x[0, 1]}")
print(f"x[:, 1]:    {x[:, 1]}")
print(f"x[1:, :2]:\n{x[1:, :2]}")

# Boolean mask
mask = x > 5
print(f"x > 5:      {x[mask]}")

# Reshape, squeeze, unsqueeze
v = torch.arange(6.0)
print(f"\nreshape(2,3):\n{v.reshape(2, 3)}")
print(f"view(3,2):\n{v.view(3, 2)}")          # view: no copy (faster)

y = torch.tensor([1.0, 2.0, 3.0])             # shape (3,)
print(f"\nunsqueeze(0): {y.unsqueeze(0).shape}")   # (1, 3)
print(f"unsqueeze(1): {y.unsqueeze(1).shape}")   # (3, 1)
print(f"squeeze:      {y.unsqueeze(0).squeeze().shape}")  # back to (3,)

# Joining tensors
a = torch.ones(2, 3)
b = torch.zeros(2, 3)
print(f"\ncat dim=0:\n{torch.cat([a, b], dim=0)}")    # stack vertically  (4,3)
print(f"cat dim=1:\n{torch.cat([a, b], dim=1)}")    # stack horizontally (2,6)
stacked = torch.stack([a, b], dim=0)              # new dim           (2,2,3)
print(f"stack shape: {stacked.shape}")

5. Autograd — Automatic Differentiation
PyTorch tracks every operation on tensors with requires_grad=True.
Calling .backward() computes all gradients automatically.
This is the engine that makes neural network training possible.

In [ ]:
# Basic autograd
x = torch.tensor(2.0, requires_grad=True)

y = x ** 3 + 2 * x             # y = x³ + 2x
y.backward()                   # compute dy/dx

# dy/dx = 3x² + 2 = 3*(4) + 2 = 14
print(f"x = {x.item()}")
print(f"y = x³ + 2x = {y.item()}")
print(f"dy/dx = 3x²+2 = {x.grad.item()}  (expected: 14.0)")

# Gradient accumulation — always zero grad before backward
x.grad.zero_()

# Vector-valued function — need to pass gradient argument
x2 = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y2 = x2 ** 2
y2.backward(torch.ones_like(x2))   # sum of gradients (common pattern)
print(f"\nx2:       {x2.data}")
print(f"dy2/dx2:  {x2.grad}         (expected: 2x = [2, 4, 6])")

# no_grad context — use during inference (no gradient tracking = faster)
with torch.no_grad():
    z = x2 ** 2
print(f"\nz.requires_grad inside no_grad: {z.requires_grad}")

# detach — create a tensor that shares data but has no grad history
detached = x2.detach()
print(f"detached.requires_grad: {detached.requires_grad}")

6. Building Models with nn.Module
nn.Module is the base class for all neural network models in PyTorch.
Define layers in __init__, define the forward pass in forward().

In [ ]:
# Option 1: nn.Sequential — quick stacking of layers
model_seq = nn.Sequential(
    nn.Linear(4, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 3),
)

# Option 2: Custom nn.Module — full control
class MLP(nn.Module):
    def __init__(self, in_features, hidden, out_features):
        super().__init__()
        self.fc1  = nn.Linear(in_features, hidden)
        self.fc2  = nn.Linear(hidden, hidden)
        self.fc3  = nn.Linear(hidden, out_features)
        self.drop = nn.Dropout(p=0.3)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)           # raw logits (no softmax here — CrossEntropyLoss handles it)
        return x

model = MLP(in_features=4, hidden=16, out_features=3)

# Inspect the model
print(model)
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total_params}")
print(f"Trainable parameters: {trainable}")

# Forward pass
x_batch = torch.randn(8, 4)             # batch of 8 samples, 4 features each
logits  = model(x_batch)               # calls model.forward(x_batch)
print(f"\nInput shape:  {x_batch.shape}")
print(f"Output shape: {logits.shape}")

7. Loss Functions
A loss function measures how wrong the model is. Lower = better.

In [ ]:
# --- Classification ---

# CrossEntropyLoss: combines LogSoftmax + NLLLoss
# Input: raw logits (NOT probabilities)
ce_loss = nn.CrossEntropyLoss()
logits  = torch.tensor([[2.0, 1.0, 0.1],
                        [0.5, 2.5, 0.3]])
targets = torch.tensor([0, 1])           # true class indices
loss_ce = ce_loss(logits, targets)
print(f"CrossEntropyLoss: {loss_ce.item():.4f}")

# BCEWithLogitsLoss: binary classification (sigmoid inside for stability)
bce_loss = nn.BCEWithLogitsLoss()
logits_b = torch.tensor([2.0, -1.0, 0.5])
targets_b = torch.tensor([1.0, 0.0, 1.0])
loss_bce  = bce_loss(logits_b, targets_b)
print(f"BCEWithLogitsLoss: {loss_bce.item():.4f}")

# --- Regression ---

# MSELoss: Mean Squared Error
mse_loss = nn.MSELoss()
preds    = torch.tensor([2.5, 0.0, 2.1])
labels   = torch.tensor([3.0, 0.5, 2.0])
loss_mse = mse_loss(preds, labels)
print(f"MSELoss:  {loss_mse.item():.4f}")

# L1Loss: Mean Absolute Error (more robust to outliers)
l1_loss  = nn.L1Loss()
loss_l1  = l1_loss(preds, labels)
print(f"L1Loss:   {loss_l1.item():.4f}")

# HuberLoss: MSE for small errors, MAE for large (best of both)
huber    = nn.HuberLoss(delta=1.0)
loss_hub = huber(preds, labels)
print(f"HuberLoss:{loss_hub.item():.4f}")

8. Optimizers
Optimizers update model weights based on computed gradients.
SGD and Adam are the two you will use 95% of the time.

In [ ]:
model = MLP(4, 16, 3)

# SGD — simple, good for CNNs with momentum
sgd = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

# Adam — adaptive, best default for most tasks
adam = optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-8)

# AdamW — Adam + decoupled weight decay (preferred over Adam for transformers)
adamw = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

# RMSprop — good for RNNs
rmsprop = optim.RMSprop(model.parameters(), lr=1e-3, alpha=0.99)

print("Optimizers initialized:")
print(f"  SGD     lr={sgd.param_groups[0]['lr']}")
print(f"  Adam    lr={adam.param_groups[0]['lr']}")
print(f"  AdamW   lr={adamw.param_groups[0]['lr']}")

# Learning rate schedulers
scheduler_step  = optim.lr_scheduler.StepLR(adam, step_size=10, gamma=0.5)     # halve LR every 10 epochs
scheduler_cos   = optim.lr_scheduler.CosineAnnealingLR(adam, T_max=50)         # cosine decay
scheduler_plat  = optim.lr_scheduler.ReduceLROnPlateau(adam, patience=5)        # reduce on plateau

print("\nLR schedulers initialized: StepLR, CosineAnnealingLR, ReduceLROnPlateau")

9. Training Loop
The standard PyTorch training loop — five steps every iteration:
zero_grad → forward → loss → backward → step

In [ ]:
# Synthetic dataset: 200 samples, 4 features, 3 classes
torch.manual_seed(42)
n_samples = 200
X_data = torch.randn(n_samples, 4)
y_data = torch.randint(0, 3, (n_samples,))

# Split into train / val
split = 160
X_train, y_train = X_data[:split], y_data[:split]
X_val,   y_val   = X_data[split:], y_data[split:]

# Model, loss, optimizer
model     = MLP(4, 16, 3)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def accuracy(logits, labels):
    preds = logits.argmax(dim=1)
    return (preds == labels).float().mean().item()

epochs = 30
for epoch in range(epochs):
    # --- Training ---
    model.train()                    # enable dropout, batch norm etc.
    optimizer.zero_grad()            # 1. clear previous gradients
    logits = model(X_train)          # 2. forward pass
    loss   = criterion(logits, y_train)  # 3. compute loss
    loss.backward()                  # 4. backward pass (compute gradients)
    optimizer.step()                 # 5. update weights

    # --- Validation (no grad) ---
    if (epoch + 1) % 5 == 0:
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val)
            val_loss   = criterion(val_logits, y_val)
        print(f"Epoch {epoch+1:3d} | "
              f"train loss: {loss.item():.4f}  acc: {accuracy(logits, y_train):.2f} | "
              f"val loss: {val_loss.item():.4f}  acc: {accuracy(val_logits, y_val):.2f}")

10. GPU Usage
Move tensors and models to GPU for fast training.
The pattern: define device, then .to(device) everything.

In [ ]:
# Detect available device
device = (
    "cuda"   if torch.cuda.is_available()  else
    "mps"    if torch.backends.mps.is_available()  else   # Apple Silicon GPU
    "cpu"
)
print(f"Using device: {device}")

# Move model and data to device
model   = MLP(4, 16, 3).to(device)
X_train = X_train.to(device)
y_train = y_train.to(device)

# Everything that touches the model must be on the same device
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss().to(device)

# One training step on device
model.train()
optimizer.zero_grad()
logits = model(X_train)
loss   = criterion(logits, y_train)
loss.backward()
optimizer.step()
print(f"Training step on {device} complete. Loss: {loss.item():.4f}")

# Move result back to CPU for numpy/display
loss_value = loss.detach().cpu().item()
print(f"Loss (CPU scalar): {loss_value:.4f}")

# Save / load model
# torch.save(model.state_dict(), 'model.pth')
# model.load_state_dict(torch.load('model.pth', map_location=device))

11. Common Layers & Activations
The building blocks you will compose into any architecture.

In [ ]:
x = torch.randn(8, 16)    # batch of 8 samples, 16 features

# --- Linear Layers ---
linear = nn.Linear(16, 8)               # fully connected: (batch, 16) → (batch, 8)
print(f"Linear:         {linear(x).shape}")

# --- Normalization ---
bn = nn.BatchNorm1d(16)                 # normalize across the batch
ln = nn.LayerNorm(16)                   # normalize across the features (used in Transformers)
print(f"BatchNorm1d:    {bn(x).shape}")
print(f"LayerNorm:      {ln(x).shape}")

# --- Dropout ---
drop = nn.Dropout(p=0.5)               # randomly zero 50% of inputs during training
print(f"Dropout:        {drop(x).shape}")

# --- Activation Functions ---
v = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
print(f"\nInput:         {v.tolist()}")
print(f"ReLU:          {F.relu(v).tolist()}")
print(f"LeakyReLU:     {F.leaky_relu(v, 0.1).tolist()}")
print(f"GELU:          {[round(x, 3) for x in F.gelu(v).tolist()]}")
print(f"Sigmoid:       {[round(x, 3) for x in torch.sigmoid(v).tolist()]}")
print(f"Tanh:          {[round(x, 3) for x in torch.tanh(v).tolist()]}")
print(f"Softmax:       {[round(x, 3) for x in F.softmax(v, dim=0).tolist()]}")

# --- Embedding Layer ---
embed = nn.Embedding(num_embeddings=100, embedding_dim=16)  # vocab of 100, 16-dim embeddings
token_ids = torch.tensor([3, 7, 42])
print(f"\nEmbedding:     {embed(token_ids).shape}")